# Phân Tích Hiệu Năng Nén KV Cache (KV Cache Compression Benchmark)

Notebook này phân tích kết quả mô phỏng nhằm so sánh hiệu năng của các thuật toán nén KV Cache khác nhau (`FP8`, `HQQ`, `PolarQuant`, `TurboQuant`) so với cấu hình cơ sở không nén (`FP16`).

## 1. Công Thức Toán Học Nền Tảng của KV Cache

Để hiểu tại sao nén KV Cache lại quan trọng, chúng ta cần xem xét công thức tính dung lượng vật lý mà KV Cache chiếm dụng trên GPU:

Dung lượng KV Cache (tính bằng Bytes) cho một sequence được tính bằng công thức:

$$ \text{KV\_Cache\_Size} = 2 \times B \times s \times L \times H \times P $$

Trong đó:
*   **$2$**: Dành cho 2 ma trận là Key ($K$) và Value ($V$).
*   **$B$**: Batch Size (Số lượng chuỗi xử lý song song).
*   **$s$**: Sequence Length (Độ dài ngữ cảnh - ví dụ 4K, 8K, 16K).
*   **$L$**: Số lượng Transformer Layers của mô hình (ví dụ Llama-3 8B có $L=32$).
*   **$H$**: Kích thước Hidden Size (ví dụ $H=4096$).
*   **$P$**: Precision (Kích thước kiểu dữ liệu tính bằng Bytes).
    *   Với **FP16** (Không nén): $P = 2$ bytes.
    *   Với **FP8**: $P = 1$ byte.
    *   Với **TurboQuant / 4-bit**: $P = 0.5$ bytes.

👉 **Như vậy:** Theo đúng công thức, khi dùng TurboQuant (4-bit), hệ số $P$ giảm từ $2$ xuống $0.5$, giúp **giảm chính xác 75%** lượng VRAM cần thiết cho KV Cache so với FP16 gốc.

## 2. Lý thuyết Nền tảng về các Thuật toán Nén KV Cache

Để giải quyết bài toán nút thắt băng thông bộ nhớ (Memory Wall), dự án này benchmark 4 kỹ thuật nén KV Cache tiên tiến nhất hiện nay. Điểm chung của chúng là làm giảm tham số $P$ (Precision) xuống, tuy nhiên cơ chế toán học bên dưới lại hoàn toàn khác nhau:

### A. Thuật toán FP8 (8-bit Floating Point)
*   **Định dạng:** Ép dữ liệu từ 16-bit xuống 8-bit chuẩn số thực dấu phẩy động (Floating Point).
*   **Cơ chế:** Cắt bỏ bớt các bit phần thập phân (Mantissa) nhưng vẫn giữ dải biểu diễn số mũ (Exponent) rộng. Đây là phương pháp tiêu chuẩn công nghiệp hiện nay, được hỗ trợ nén/giải nén trực tiếp bằng phần cứng trên các dòng card đời mới (như NVIDIA Hopper H100).
*   **Đặc điểm:** Giảm một nửa RAM ($P=1$) với tỷ lệ hao hụt chất lượng (PPL) gần như bằng không.

### B. Thuật toán HQQ (Half-Quadratic Quantization)
*   **Định dạng:** Thường ép xuống dải 4-bit hoặc 8-bit nguyên (Integer).
*   **Cơ chế:** Là một phương pháp nén toán học không cần bộ dữ liệu hiệu chuẩn (Data-free / Calibration-free). HQQ sử dụng thuật toán tối ưu hóa Half-Quadratic để tìm ra tỷ lệ thu phóng (Scale) và điểm 0 (Zero-point) hoàn hảo nhất cực kỳ nhanh chóng.
*   **Đặc điểm:** Nén nhanh, tính tương thích cao, hoạt động tốt trên nhiều dải tham số mà không cần mất hàng giờ để Fine-tune bộ nén.

### C. Thuật toán PolarQuant
*   **Định dạng:** Nén dải 4-bit dựa trên tọa độ.
*   **Cơ chế:** Nhận thấy các Outliers (các giá trị ngoại lai cực lớn trong KV Cache) rất khó nén, PolarQuant sử dụng phép biến đổi không gian (thường chuyển sang tọa độ cực - Polar Coordinates hoặc phân cực giá trị) để cô lập các Outliers này, từ đó nén phần dữ liệu cốt lõi xuống 4-bit một cách êm ái mà không làm biến dạng cấu trúc gốc.
*   **Đặc điểm:** Giữ được độ chính xác (PPL thấp) cực tốt ngay cả khi nén sâu, nhưng thuật toán phức tạp hơn dẫn đến Latency có thể nhích nhẹ.

### D. Thuật toán TurboQuant (Tối ưu hóa phần cứng cực đại)
*   **Định dạng:** Ép cực đoan xuống 4-bit ($P=0.5$).
*   **Cơ chế:** Khác với các thuật toán chỉ tập trung vào toán học, TurboQuant tập trung vào kỹ thuật lập trình phần cứng. Nó sở hữu các nhân tính toán (CUDA/Triton Kernels) viết bằng ngôn ngữ bậc thấp, cho phép luồng Attention GPU nạp trực tiếp luồng bit 4-bit từ bộ nhớ và giải nén (Dequantize) "on-the-fly" ngay trên thanh ghi (Registers) siêu tốc.
*   **Hiệu ứng "Turbo":** Vừa giảm được 75% VRAM (chống OOM tuyệt đối), vừa vượt qua được nút thắt Memory Wall, khiến chi phí giải nén gần như bị triệt tiêu bởi tốc độ chép dữ liệu khổng lồ được giải phóng.

## 3. Phương pháp luận & Công thức đo lường Thực Nghiệm

Từ công thức nền tảng ở trên, chúng ta đo đạc 3 trụ cột thực tế:

### A. Hiệu quả tiết kiệm Peak VRAM
Peak VRAM bao gồm Model Weights + KV Cache + Activation Memory. Phần trăm tiết kiệm được tính:

$$ \text{VRAM Reduction (\%)} = \left( 1 - \frac{\text{Peak VRAM}_{\text{Method}}}{\text{Peak VRAM}_{\text{FP16}}} \right) \times 100 $$

### B. Độ trễ và Thông lượng (Latency & Throughput)
* **Latency Overhead:** Độ trễ tăng thêm do việc giải nén (Dequantize) tensor $K$ và $V$ từ 4-bit lên 16-bit ở mỗi layer trong quá trình attention.
* **Throughput ($tokens/s$):** $1000 / \text{Latency (ms/token)}$.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Thiết lập style đồ thị chuyên nghiệp
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams.update({'figure.dpi': 120, 'font.size': 12})

# Tải dữ liệu
df = pd.read_csv('template_log.csv')
df['peak_memory_mb'] = pd.to_numeric(df['peak_memory_mb'], errors='coerce')
df['latency_ms_per_token'] = pd.to_numeric(df['latency_ms_per_token'], errors='coerce')
df['throughput_tokens_per_s'] = pd.to_numeric(df['throughput_tokens_per_s'], errors='coerce')

display(df.head())

## 4. So sánh Peak VRAM trên các Mô hình ở Context 16K
Biểu đồ dưới đây thể hiện mức sử dụng bộ nhớ Peak VRAM của 5 mô hình ngôn ngữ khi đạt tới ngữ cảnh tối đa 16000 tokens. Thanh TurboQuant thấp nhất thể hiện việc $P=0.5$ đã phát huy tác dụng.

In [ ]:
df_16k = df[df['context_length'] == 16000].copy()
plt.figure(figsize=(14, 7))
ax = sns.barplot(x="model", y="peak_memory_mb", hue="kv_cache_type", data=df_16k)
plt.title('Peak VRAM Consumption across Models (Context Length = 16K)', fontsize=16, fontweight='bold')
plt.ylabel('Peak Memory (MB)')
plt.xlabel('Model')
plt.xticks(rotation=15)
plt.legend(title='KV Cache Type')
plt.tight_layout()
plt.show()

## 5. Sự gia tăng VRAM theo Context Length (PhoGPT-7B5)
Đường biểu diễn $y = \text{Base} + \text{KV\_Cache\_Size(s)}$. Vì hàm số phụ thuộc bậc 1 vào $s$ (Sequence Length), ta thấy các đường đều là đường thẳng tuyến tính. Tuy nhiên, hệ số góc của đường TurboQuant nhỏ hơn 4 lần so với FP16 do hệ số $P$ nhỏ hơn.

In [ ]:
df_pho = df[df['model'] == 'VinAI/PhoGPT-7B5-Instruct'].copy()
plt.figure(figsize=(10, 6))
sns.lineplot(x='context_length', y='peak_memory_mb', hue='kv_cache_type', 
             marker='o', linewidth=2.5, markersize=8, data=df_pho)
plt.title('VRAM Scaling with Context Length (PhoGPT-7B5)', fontsize=14, fontweight='bold')
plt.ylabel('Peak Memory (MB)')
plt.xlabel('Context Length (Tokens)')
plt.xticks([4000, 8000, 16000])
plt.grid(True, linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

## 6. Pareto Frontier (Đánh đổi giữa Memory và Latency)
Đồ thị Scatter Plot thể hiện Trade-off.

In [ ]:
plt.figure(figsize=(10, 6))
df_pareto = df[(df['model'] == 'meta-llama/Meta-Llama-3.1-8B-Instruct') & (df['context_length'] == 16000)]
sns.scatterplot(x='peak_memory_mb', y='latency_ms_per_token', hue='kv_cache_type', 
                s=200, edgecolor='black', alpha=0.8, data=df_pareto)
for i, row in df_pareto.iterrows():
    plt.text(row['peak_memory_mb'] + 100, row['latency_ms_per_token'], 
             row['kv_cache_type'], fontsize=10)
plt.title('Memory vs Latency Trade-off (Llama-3.1-8B @ 16K)', fontsize=14, fontweight='bold')
plt.xlabel('Peak Memory (MB) -> (Càng nhỏ càng tốt)')
plt.ylabel('Latency (ms/token) -> (Càng nhỏ càng tốt)')
plt.tight_layout()
plt.show()